In [ ]:
import requests
import pandas as pd
from io import BytesIO
import time
from datetime import datetime

def download_airportal_data(year, month):
    """
    에어포탈에서 항공사별 데이터를 다운로드하여 데이터프레임으로 반환

    Parameters:
    year (int): 조회 년도 (예: 2022)
    month (int): 조회 월 (예: 1)

    Returns:
    pd.DataFrame: 항공사별 통계 데이터
    """

    base_url = "https://www.airportal.go.kr"
    excel_url = f"{base_url}/stats/transport/getDetailedAirTransportStats1Excel.do"

    month_str = str(month).zfill(2)
    year_month = f"{year}{month_str}"

    params = {
        'last_yearmonth': year_month,
        'this_yearmonth': year_month,
        'pass_gubun': '4',
        'carge_gubun': 'total',
        'sn_gubun': 'total',
        'airline_gubun': 'total',
        'di_gubun': 'total',
        'pyn_gubun': 'total',
        'arvl_type': 'total'
    }

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
        'Content-Type': 'application/json;charset=UTF-8',
        'Referer': f'{base_url}/stats/transport/chartDetail.do'
    }

    try:
        response = requests.post(
            excel_url,
            json=params,
            headers=headers,
            timeout=30
        )

        if response.status_code == 200:
            excel_file = BytesIO(response.content)
            df = pd.read_excel(excel_file, engine='openpyxl')

            # 빈 컬럼 제거
            df = df.dropna(axis=1, how='all')
            df = df.loc[:, ~df.columns.str.contains('^Unnamed', na=False)]

            # 조회 기준 정보 추가
            df['년도'] = year
            df['월'] = month
            df['년월'] = f"{year}-{month:02d}"

            print(f"✓ {year}년 {month}월 완료 (행수: {len(df)})")

            return df
        else:
            print(f"✗ {year}년 {month}월 실패: HTTP {response.status_code}")
            return None

    except Exception as e:
        print(f"✗ {year}년 {month}월 오류: {str(e)}")
        return None


def download_multiple_years(start_year, start_month, end_year, end_month, delay=1):
    """
    여러 해의 데이터를 월별로 다운로드

    Parameters:
    start_year (int): 시작 년도
    start_month (int): 시작 월
    end_year (int): 종료 년도
    end_month (int): 종료 월
    delay (float): 각 요청 사이 대기 시간(초)

    Returns:
    pd.DataFrame: 결합된 데이터프레임
    """
    all_data = []
    failed_months = []

    current_year = start_year
    current_month = start_month

    # 총 개월 수 계산
    total_months = (end_year - start_year) * 12 + (end_month - start_month) + 1
    processed = 0

    print(f"\n{'='*60}")
    print(f"다운로드 시작: {start_year}년 {start_month}월 ~ {end_year}년 {end_month}월")
    print(f"총 {total_months}개월 데이터 수집 예정")
    print('='*60)

    while (current_year < end_year) or (current_year == end_year and current_month <= end_month):
        processed += 1
        print(f"\n[{processed}/{total_months}] ", end="")

        df = download_airportal_data(current_year, current_month)

        if df is not None and not df.empty:
            all_data.append(df)
        else:
            failed_months.append(f"{current_year}-{current_month:02d}")

        # 다음 달로 이동
        current_month += 1
        if current_month > 12:
            current_month = 1
            current_year += 1

        # 서버 부하 방지를 위한 대기
        if processed < total_months:  # 마지막이 아니면 대기
            time.sleep(delay)

    print(f"\n{'='*60}")
    print(f"다운로드 완료!")
    print(f"성공: {len(all_data)}개월, 실패: {len(failed_months)}개월")

    if failed_months:
        print(f"실패한 월: {', '.join(failed_months)}")

    # 모든 데이터 결합
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"총 데이터 행 수: {len(combined_df):,}")
        print('='*60)
        return combined_df
    else:
        print("수집된 데이터가 없습니다.")
        return None


def download_full_years(years, delay=1):
    """
    특정 연도들의 1~12월 전체 데이터 다운로드

    Parameters:
    years (list): 다운로드할 연도 리스트 (예: [2020, 2021, 2022])
    delay (float): 각 요청 사이 대기 시간(초)

    Returns:
    pd.DataFrame: 결합된 데이터프레임
    """
    all_data = []

    for year in years:
        print(f"\n{'='*60}")
        print(f"{year}년 전체 데이터 다운로드 시작")
        print('='*60)

        year_data = download_multiple_years(year, 1, year, 12, delay)

        if year_data is not None:
            all_data.append(year_data)
            print(f"\n{year}년 완료: {len(year_data):,} 행")

    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        return None


# 사용 예시
# if __name__ == "__main__":
#
#     # 방법 1: 특정 기간 다운로드 (2020년 1월 ~ 2022년 12월)
#     print("\n" + "="*60)
#     print("방법 1: 특정 기간 다운로드")
#     print("="*60)
#
#     df_period = download_multiple_years(
#         start_year=2020,
#         start_month=1,
#         end_year=2022,
#         end_month=12,
#         delay=1  # 1초 대기
#     )
#
#     if df_period is not None:
#         # 저장
#         output_file = 'airportal_2020_2022.csv'
#         df_period.to_csv(output_file, index=False, encoding='utf-8-sig')
#         print(f"\n데이터가 '{output_file}'로 저장되었습니다.")
#
#         # 요약 통계
#         print("\n=== 연도별 데이터 건수 ===")
#         print(df_period.groupby('년도').size())
#
#         print("\n=== 데이터 미리보기 ===")
#         print(df_period.head(10))
#
#
#     # 방법 2: 여러 연도의 전체 데이터 다운로드
#     print("\n\n" + "="*60)
#     print("방법 2: 여러 연도 전체 다운로드")
#     print("="*60)
#
#     df_years = download_full_years(
#         years=[2020, 2021, 2022, 2023],
#         delay=1
#     )
#
#     if df_years is not None:
#         # 저장
#         output_file = 'airportal_2020_2023_full.csv'
#         df_years.to_csv(output_file, index=False, encoding='utf-8-sig')
#         print(f"\n데이터가 '{output_file}'로 저장되었습니다.")
#
#         # 연도별, 월별 요약
#         print("\n=== 연도별 데이터 건수 ===")
#         print(df_years.groupby('년도').size())
#
#         print("\n=== 월별 평균 데이터 건수 ===")
#         print(df_years.groupby('월').size().mean())
#
#         # 엑셀로도 저장 (시트별로 연도 구분)
#         excel_file = 'airportal_2020_2023_full.xlsx'
#         with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
#             # 전체 데이터
#             df_years.to_excel(writer, sheet_name='전체', index=False)
#
#             # 연도별 시트
#             for year in df_years['년도'].unique():
#                 year_df = df_years[df_years['년도'] == year]
#                 year_df.to_excel(writer, sheet_name=str(year), index=False)
#
#         print(f"엑셀 파일도 '{excel_file}'로 저장되었습니다.")
#
#
#     # 방법 3: 최근 N개월 다운로드
#     print("\n\n" + "="*60)
#     print("방법 3: 최근 24개월 다운로드")
#     print("="*60)
#
#     # 현재 날짜 기준으로 24개월 전부터 다운로드
#     now = datetime.now()
#     end_year = now.year
#     end_month = now.month - 1  # 전월까지
#
#     if end_month < 1:
#         end_month = 12
#         end_year -= 1
#
#     start_year = end_year - 2  # 2년 전
#     start_month = end_month
#
#     df_recent = download_multiple_years(
#         start_year=start_year,
#         start_month=start_month,
#         end_year=end_year,
#         end_month=end_month,
#         delay=1
#     )
#
#     if df_recent is not None:
#         output_file = f'airportal_recent_{start_year}{start_month:02d}_{end_year}{end_month:02d}.csv'
#         df_recent.to_csv(output_file, index=False, encoding='utf-8-sig')
#         print(f"\n최근 24개월 데이터가 '{output_file}'로 저장되었습니다.")

In [ ]:
df = download_multiple_years(2015, 1, 2025, 12)# df_2022_01 = download_airportal_data(2022, 1)

In [14]:
# path = r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KS_Air_Tourism'
# df.to_csv(r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KS_Air_Tourism\airportal_sample.csv')